In [41]:
#!pip install trafilatura justext beautifulsoup4 lxml pandas

import re
import html
import pandas as pd
from bs4 import BeautifulSoup
import trafilatura
import justext
import numpy as np
import unicodedata
import requests
from typing import List, Dict

from concurrent.futures import ThreadPoolExecutor, as_completed

import os
import json
import time
from sqlalchemy import create_engine, text
import math

from tqdm.auto import tqdm
tqdm.pandas()

### Cleansing - Parsing Pipeline Jobs

In [2]:
def normalize_raw_text(raw):
    if raw is None:
        return ""

    text = str(raw).strip()

    text = text.strip("'").strip('"')

    text = re.sub(r"^\s*Text:\s*", "", text, flags=re.IGNORECASE)

    text = html.unescape(text)

    return text.strip()

def extract_with_trafilatura(text):
    if not text.strip():
        return ""

    extracted = trafilatura.extract(
        text,
        output_format="txt",
        include_comments=False,
        include_tables=False,
        include_links=False,
        favor_precision=True
    )

    return (extracted or "").strip()

def extract_with_justext(text):
    if not text.strip():
        return ""

    paragraphs = justext.justext(text, justext.get_stoplist("English"))

    cleaned_parts = []
    for p in paragraphs:
        if not p.is_boilerplate:
            cleaned_parts.append(p.text.strip())

    return "\n".join(part for part in cleaned_parts if part).strip()

def extract_with_bs4(text):
    if not text.strip():
        return ""

    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = re.sub(r"(?i)</p\s*>", "\n", text)
    text = re.sub(r"(?i)<p\s*>", "", text)
    text = re.sub(r"(?i)</li\s*>", "\n", text)
    text = re.sub(r"(?i)<li\s*>", "- ", text)

    soup = BeautifulSoup(text, "lxml")
    extracted = soup.get_text(separator="\n")

    return extracted.strip()

SECTION_HEADERS = [
    "Purpose of the role:",
    "Main activities:",
    "Minimum requirements:",
    "Availability:",
    "Responsibilities:",
    "Requirements:",
    "Qualifications:",
    "Preferred qualifications:",
    "About the role:",
    "What you will do:",
    "What we're looking for:"
]

def clean_extracted_text(text):
    if not text:
        return ""

    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n[ \t]*[-•]\s*", "\n- ", text)

    for header in SECTION_HEADERS:
        pattern = re.escape(header)
        text = re.sub(pattern, f"\n\n{header}\n", text, flags=re.IGNORECASE)

    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

def parse_job_text(raw, min_length):
    normalized = normalize_raw_text(raw)

    if not normalized:
        return ""

    extracted = extract_with_trafilatura(normalized)

    if len(extracted) < min_length:
        extracted = extract_with_justext(normalized)

    if len(extracted) < min_length:
        extracted = extract_with_bs4(normalized)

    cleaned = clean_extracted_text(extracted)

    return cleaned

import re

def to_natural_text(text):
    if not text:
        return ""

    out = []

    for line in text.split("\n"):
        line = line.strip()

        if not line:
            continue

        line = re.sub(r"^[-•*]\s*", "", line)

        line = re.sub(r"\s+([,.;:!?])", r"\1", line)

        if line and not re.search(r"[.!?:]$", line):
            line += "."

        out.append(line)

    natural = " ".join(out)
    natural = re.sub(r"\s+", " ", natural).strip()

    return natural

In [3]:
df_matching_jp = pd.read_excel(r"C:\Users\anton\Desktop\Repos\LabAI\Data\Matching\Matching_training.xlsx",sheet_name="Job_posts")
df_matching_cd = pd.read_excel(r"C:\Users\anton\Desktop\Repos\LabAI\Data\Matching\Matching_training.xlsx",sheet_name="Candidates")

In [4]:
df_matching_jp['Job Post'] = df_matching_jp['Job_post_eng'].apply(lambda x: parse_job_text(x,80))
df_matching_jp['Job Post Natural'] = df_matching_jp['Job Post'].apply(lambda x: to_natural_text(x))

### Cleansing - Parsing Pipeline Candidates

In [ ]:
def normalize_cv_raw(raw):
    if raw is None:
        return ""

    text = str(raw).strip()
    text = html.unescape(text)

    text = re.sub(r"^\s*['\"]?\s*Anonymized CV:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*Text:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*CV:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r'"""+', "", text)

    text = text.strip("'").strip('"')
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def extract_cv_text_basic(text):
    if not text.strip():
        return ""

    if "<" in text and ">" in text:
        text = re.sub(r"(?i)<br\s*/?>", "\n", text)
        text = re.sub(r"(?i)</p\s*>", "\n", text)
        text = re.sub(r"(?i)<p\s*>", "", text)
        text = re.sub(r"(?i)</li\s*>", "\n", text)
        text = re.sub(r"(?i)<li\s*>", "- ", text)

        soup = BeautifulSoup(text, "lxml")
        return soup.get_text(separator="\n").strip()

    return text.strip()

def extract_cv_text_fallback(text):
    if not text.strip():
        return ""

    text = text.replace("\\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

CV_SECTION_HEADERS = [
    "PROFILE",
    "EDUCATION",
    "PERSONAL DATA",
    "PROFESSIONAL EXPERIENCE",
    "WORK EXPERIENCE",
    "SKILLS",
    "LANGUAGES",
    "HOBBIES",
    "INTERNSHIP",
    "VOLUNTEERING"
]

def clean_cv_text(text):
    if not text:
        return ""

    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(
        r"I authorize the processing of my personal data.*$",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(r"\n[ \t]*[-•]\s*", "\n- ", text)

    for header in CV_SECTION_HEADERS:
        pattern = re.escape(header)
        text = re.sub(rf"(?im)^\s*{pattern}\s*$", f"\n\n{header}\n", text)

    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

def parse_cv_text(raw, min_length: int = 80):
    normalized = normalize_cv_raw(raw)

    if not normalized:
        return ""

    extracted = extract_cv_text_basic(normalized)

    if len(extracted) < min_length:
        extracted = extract_cv_text_fallback(normalized)

    cleaned = clean_cv_text(extracted)

    return cleaned

def clean_text_for_pg(x):
    if pd.isna(x) or x is None:
        return None

    x = str(x)

    replacements = {
        "\u25aa": "-",
        "\u2022": "-",
        "\u25cf": "-",
        "\u25e6": "-",
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": ""
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    x = x.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
    return x.strip()

def clean_cv_text_strong(x):
    if x is None or pd.isna(x):
        return None

    x = str(x)

    x = unicodedata.normalize("NFKC", x)

    x = x.replace("\r\n", "\n").replace("\r", "\n")

    replacements = {
        "\u25aa": "-",
        "\u2022": "-",
        "\u25cf": "-",
        "\u25e6": "-",
        "\u2043": "-",
        "\u2219": "-",
        "\u00b7": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u00a0": " ",
        "\u2000": " ",
        "\u2001": " ",
        "\u2002": " ",
        "\u2003": " ",
        "\u2004": " ",
        "\u2005": " ",
        "\u2006": " ",
        "\u2007": " ",
        "\u2008": " ",
        "\u2009": " ",
        "\u200a": " ",
        "\u200b": "",
        "\u200c": "",
        "\u200d": "",
        "\u2060": "",
        "\ufeff": "",
        "\u2028": "\n",
        "\u2029": "\n"
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    x = "".join(ch for ch in x if unicodedata.category(ch)[0] != "C" or ch in "\n\t")

    x = re.sub(r"[^\S\n]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)

    x = x.encode("cp1252", errors="ignore").decode("cp1252", errors="ignore")

    x = re.sub(r"[ ]{2,}", " ", x)
    x = re.sub(r"\n +", "\n", x)

    return x.strip()

In [6]:
df_matching_cd['CV Clean'] = df_matching_cd['CV_text'].apply(lambda x: parse_cv_text(x,80))
df_matching_cd['CV Natural'] = df_matching_cd['CV Clean'].apply(lambda x: to_natural_text(x))

In [23]:
df_matching_cd_short = df_matching_cd.iloc[:100,:].copy()

In [24]:
df_matching_jp_short = df_matching_jp.iloc[500:600,:].copy()

In [ ]:
df_matching_jp_short["Job Post Natural"] = df_matching_jp_short["Job Post Natural"].apply(clean_text_for_pg)
df_matching_cd_short["CV Natural"] = df_matching_cd_short["CV Natural"].apply(clean_cv_text_strong)

### Extract Info

In [82]:
import requests

SPACE_BASE_URL = "https://bard53-quenextractor.hf.space/extract_profile"

def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"^\s*Text:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

import json
import re
import requests


def strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def try_close_json(candidate: str) -> str:
    open_braces = candidate.count("{")
    close_braces = candidate.count("}")
    if close_braces < open_braces:
        candidate += "}" * (open_braces - close_braces)
    return candidate


def extract_json_block(text: str) -> dict:
    text = strip_code_fences(text)

    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found")

    candidate = text[start:].strip()
    candidate = try_close_json(candidate)
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)

    try:
        return json.loads(candidate)
    except Exception:
        pass

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i + 1]
                candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
                return json.loads(candidate)

    raise ValueError("No balanced JSON object found")


def clean_scalar(value) -> str:
    if value is None:
        return ""

    s = str(value).strip()

    invalid_values = {
        "n/a",
        "na",
        "none",
        "null",
        "unknown",
        "not specified",
        "not provided",
        "-"
    }

    if s.lower() in invalid_values:
        return ""

    return s


def to_string_list(value) -> list[str]:
    if value is None:
        return []

    if isinstance(value, list):
        out = []
        for v in value:
            if isinstance(v, str):
                s = v.strip()
                if s:
                    out.append(s)
            elif v is not None:
                s = str(v).strip()
                if s:
                    out.append(s)
        return list(dict.fromkeys(out))

    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []

    s = str(value).strip()
    return [s] if s else []


def normalize_experiences(value) -> list[str]:
    if value is None:
        return []

    out = []

    if isinstance(value, list):
        for item in value:
            if isinstance(item, str):
                s = item.strip()
                if s:
                    out.append(s)

            elif isinstance(item, dict):
                title = str(item.get("title", "")).strip()
                from_date = str(item.get("from_date", "")).strip()
                to_date = str(item.get("to_date", "")).strip()

                if title and from_date and to_date:
                    out.append(f"{title} ({from_date} - {to_date})")
                elif title and from_date:
                    out.append(f"{title} ({from_date})")
                elif title:
                    out.append(title)

            elif item is not None:
                s = str(item).strip()
                if s:
                    out.append(s)

    elif isinstance(value, str):
        s = value.strip()
        if s:
            out.append(s)

    return list(dict.fromkeys(out))


def normalize_profile(profile: dict) -> dict:
    if not isinstance(profile, dict):
        profile = {}

    job_title = clean_scalar(profile.get("job_title", ""))
    skills = to_string_list(profile.get("skills", []))
    experiences = normalize_experiences(profile.get("experiences", []))
    location = clean_scalar(profile.get("location", ""))
    summary = clean_scalar(profile.get("summary", ""))

    if not job_title and experiences:
        job_title = experiences[0].strip()

    return {
        "job_title": job_title,
        "skills": skills,
        "experiences": experiences,
        "location": location,
        "summary": summary,
    }


def salvage_profile_from_raw(raw_output: str) -> dict:
    text = strip_code_fences(raw_output)

    job_title_match = re.search(r'"job_title"\s*:\s*"([^"]*)"', text, flags=re.DOTALL)
    location_match = re.search(r'"location"\s*:\s*"([^"]*)"', text, flags=re.DOTALL)
    summary_match = re.search(r'"summary"\s*:\s*"([^"]*)', text, flags=re.DOTALL)

    skills = []
    skills_block = re.search(r'"skills"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)
    if skills_block:
        skills = re.findall(r'"([^"]*)"', skills_block.group(1))

    experiences = []
    experiences_block = re.search(r'"experiences"\s*:\s*\[(.*?)]\s*,\s*"location"', text, flags=re.DOTALL)
    if experiences_block:
        exp_block = experiences_block.group(1)

        dict_titles = re.findall(
            r'\{\s*"title"\s*:\s*"([^"]*)"(?:\s*,\s*"from_date"\s*:\s*"([^"]*)")?(?:\s*,\s*"to_date"\s*:\s*"([^"]*)")?.*?\}',
            exp_block,
            flags=re.DOTALL
        )
        if dict_titles:
            for title, from_date, to_date in dict_titles:
                title = title.strip()
                from_date = from_date.strip()
                to_date = to_date.strip()

                if title and from_date and to_date:
                    experiences.append(f"{title} ({from_date} - {to_date})")
                elif title and from_date:
                    experiences.append(f"{title} ({from_date})")
                elif title:
                    experiences.append(title)
        else:
            experiences = re.findall(r'"([^"]*)"', exp_block)

    profile = {
        "job_title": job_title_match.group(1).strip() if job_title_match else "",
        "skills": skills,
        "experiences": experiences,
        "location": location_match.group(1).strip() if location_match else "",
        "summary": summary_match.group(1).strip() if summary_match else "",
    }

    return normalize_profile(profile)


def extract_profile(text: str, document_type: str = "cv", timeout: int = 120):
    cleaned = clean_text(text)

    payload = {
        "text": cleaned,
        "document_type": document_type
    }

    resp = requests.post(SPACE_BASE_URL, json=payload, timeout=timeout)

    if resp.ok:
        data = resp.json()
        return normalize_profile(data["profile"])

    print("STATUS:", resp.status_code)

    try:
        err = resp.json()
        print("ERROR BODY:", err)
    except Exception:
        print("RAW RESPONSE TEXT:\n", resp.text)
        resp.raise_for_status()

    detail = err.get("detail", {})

    if resp.status_code == 422 and isinstance(detail, dict):
        raw_output = detail.get("raw_output")
        if raw_output:
            print("SERVER ERROR:", detail.get("error"))
            print("RAW OUTPUT:\n", raw_output)

            try:
                parsed = extract_json_block(raw_output)
                return normalize_profile(parsed)
            except Exception:
                return salvage_profile_from_raw(raw_output)

    resp.raise_for_status()

In [83]:
profile = extract_profile(
    text=df_matching_cd_short.iloc[0]["CV Natural"],
    document_type="cv"
)

profile

{'job_title': 'Volunteering at Concresco Association',
 'skills': ['communication', 'problem-solving', 'secretarial'],
 'experiences': ['Volunteering at Concresco Association',
  'Internship at CAF Italia'],
 'location': 'Sassuolo, Italy',
 'summary': 'Graduated from Elsa Morante Institute, highly motivated and skilled in collaboration and communication.'}

### Embed columns

In [25]:
URL = "https://bard53-mmbert.hf.space/embed"

def get_embedding(text, timeout=60):
    response = requests.post(
        URL,
        json={"text": text},
        timeout=timeout,
        verify=False
    )

    if response.status_code != 200:
        raise Exception(f"Error {response.status_code}: {response.text}")

    return response.json()["embedding"]

def parallel_get_embeddings(series, max_workers=4):
    texts = series.tolist()
    results = [None] * len(texts)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(get_embedding, text): i
            for i, text in enumerate(texts)
        }

        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx)):
            i = future_to_idx[future]
            try:
                results[i] = future.result()
            except Exception as e:
                results[i] = None
                print(f"Errore riga {i}: {e}")

    return results

In [26]:
df_matching_cd_short["Embedding"] = parallel_get_embeddings(
    df_matching_cd_short["CV Natural"],
    max_workers=10
)

 50%|█████     | 50/100 [01:02<00:42,  1.19it/s]

Errore riga 58: Error 400: {"detail":"Empty text"}


100%|██████████| 100/100 [01:54<00:00,  1.14s/it]


In [27]:
df_matching_cd_short = df_matching_cd_short[df_matching_cd_short['CV Natural'] != ""].copy()

In [28]:
df_matching_jp_short["Embedding"] = parallel_get_embeddings(
    df_matching_jp_short["Job Post Natural"],
    max_workers=10
)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [01:09<00:00,  1.44it/s]


### Populate DB

In [44]:
PG_USER = "postgres"
PG_PASSWORD = "tartosso2"
PG_HOST = "localhost"
PG_PORT = "5433"
PG_DB = "vectordb"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    future=True
)

In [45]:
def ensure_float_list(x):
    if x is None:
        return None

    if isinstance(x, np.ndarray):
        x = x.tolist()

    if not isinstance(x, list):
        return None

    out = []
    for v in x:
        if v is None:
            return None
        fv = float(v)
        if math.isnan(fv) or math.isinf(fv):
            return None
        out.append(fv)

    return out


def to_pgvector(x):
    x = ensure_float_list(x)
    if x is None:
        return None
    return "[" + ",".join(str(float(v)) for v in x) + "]"


def infer_vector_dim(df, emb_col="Embedding"):
    for x in df[emb_col]:
        x = ensure_float_list(x)
        if x is not None:
            return len(x)
    raise ValueError(f"Nessun embedding valido trovato nella colonna {emb_col}")

In [46]:
jobs = df_matching_jp_short.copy()
cvs = df_matching_cd_short.copy()

jobs["Embedding"] = jobs["Embedding"].apply(ensure_float_list)
cvs["Embedding"] = cvs["Embedding"].apply(ensure_float_list)

jobs = jobs[jobs["Embedding"].notna()].copy()
cvs = cvs[cvs["Embedding"].notna()].copy()

jobs["Embedding_pg"] = jobs["Embedding"].apply(to_pgvector)
cvs["Embedding_pg"] = cvs["Embedding"].apply(to_pgvector)

job_dim = infer_vector_dim(jobs, "Embedding")
cv_dim = infer_vector_dim(cvs, "Embedding")

print("job_dim:", job_dim)
print("cv_dim:", cv_dim)

if job_dim != cv_dim:
    raise ValueError(f"Dimensione embedding diversa: jobs={job_dim}, cvs={cv_dim}")

job_dim: 768
cv_dim: 768


In [47]:
with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))

In [48]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS job_embeddings"))
    conn.execute(text(f"""
        CREATE TABLE job_embeddings (
            Offer_ID TEXT PRIMARY KEY,
            job_post_natural TEXT,
            embedding vector({job_dim})
        )
    """))

    conn.execute(text("DROP TABLE IF EXISTS cv_embeddings"))
    conn.execute(text(f"""
        CREATE TABLE cv_embeddings (
            candidate_id TEXT PRIMARY KEY,
            cv_natural TEXT,
            embedding vector({cv_dim})
        )
    """))

In [49]:
job_records = []
for _, row in jobs.iterrows():
    job_records.append({
        "offer_id": str(row["Offer_ID"]),
        "job_post_natural": row["Job Post Natural"] if pd.notna(row["Job Post Natural"]) else None,
        "embedding_pg": row["Embedding"]
    })

len(job_records)

100

In [50]:
cv_records = []
for _, row in cvs.iterrows():
    cv_records.append({
        "candidate_id": str(row["CANDIDATE_ID"]),
        "cv_natural": row["CV Natural"] if pd.notna(row["CV Natural"]) else None,
        "embedding_pg": row["Embedding"]
    })

len(cv_records)

99

In [51]:
insert_jobs_sql = text("""
    INSERT INTO job_embeddings (
        Offer_ID,
        job_post_natural,
        embedding
    )
    VALUES (
        :offer_id,
        :job_post_natural,
        CAST(:embedding_pg AS vector)
    )
""")

insert_cvs_sql = text("""
    INSERT INTO cv_embeddings (
        candidate_id,
        cv_natural,
        embedding
    )
    VALUES (
        :candidate_id,
        :cv_natural,
        CAST(:embedding_pg AS vector)
    )
""")

In [52]:
with engine.begin() as conn:
    conn.execute(insert_jobs_sql, job_records)
    conn.execute(insert_cvs_sql, cv_records)

In [66]:
with engine.begin() as conn:
    conn.execute(text("DROP INDEX IF EXISTS idx_job_embeddings_cosine"))
    conn.execute(text("DROP INDEX IF EXISTS idx_cv_embeddings_cosine"))

with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_job_embeddings_hnsw_cosine
        ON job_embeddings
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64)
    """))

    conn.execute(text("""
        CREATE INDEX IF NOT EXISTS idx_cv_embeddings_hnsw_cosine
        ON cv_embeddings
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64)
    """))

    conn.execute(text("ANALYZE job_embeddings"))
    conn.execute(text("ANALYZE cv_embeddings"))

In [67]:
def search_jobs(query_embedding, k=5, ef_search=100):
    query_vec = to_pgvector(query_embedding)

    sql = text("""
        SELECT
            Offer_ID,
            job_post_natural,
            1 - (embedding <=> CAST(:query_vec AS vector)) AS cosine_similarity
        FROM job_embeddings
        ORDER BY embedding <=> CAST(:query_vec AS vector)
        LIMIT :k
    """)

    with engine.begin() as conn:
        conn.execute(text("SET LOCAL hnsw.ef_search = :ef"), {"ef": ef_search})
        rows = conn.execute(sql, {"query_vec": query_vec, "k": k}).mappings().all()

    return pd.DataFrame(rows)


def search_cvs(query_embedding, k=5, ef_search=100):
    query_vec = to_pgvector(query_embedding)

    sql = text("""
        SELECT
            candidate_id,
            cv_natural,
            1 - (embedding <=> CAST(:query_vec AS vector)) AS cosine_similarity
        FROM cv_embeddings
        ORDER BY embedding <=> CAST(:query_vec AS vector)
        LIMIT :k
    """)

    with engine.begin() as conn:
        conn.execute(text("SET LOCAL hnsw.ef_search = :ef"), {"ef": ef_search})
        rows = conn.execute(sql, {"query_vec": query_vec, "k": k}).mappings().all()

    return pd.DataFrame(rows)

In [70]:
cand_emb = cvs.iloc[2]["Embedding"]
top_job = search_jobs(cand_emb, k=50)
top_job

,offer_id,job_post_natural,cosine_similarity
0,424_3312,ABC company Italia Spa is looking for a CANTEE...,0.922843
1,424_3313,ABC company Italia Spa is looking for a CANTEE...,0.920347
2,424_3311,ABC company Italia Spa is looking for a CANTEE...,0.910821
3,397_4619,For a company operating in interior renovation...,0.900187
4,413_1002,"Company operating in the metalworking sector, ...",0.889488
5,413_1003,We are looking for an Electrician Worker for a...,0.889480
6,456_2625,We are looking for dining room staff for a lea...,0.889046
7,431_4850,ABC company Italia Spa is looking for an emplo...,0.888804
8,441_4758,For a prestigious company located in Valeggio ...,0.887725
9,457_2903,Do you have experience as a CNC Lathe Operator...,0.886741
